In [14]:
import pandas as pd
import numpy as np
import evaluate
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from datasets import DatasetDict, Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,classification_report)
from transformers import EarlyStoppingCallback

In [4]:
torch.cuda.is_available()

True

In [6]:
df = pd.read_csv("/kaggle/input/rucola/train_rucola.csv")  
df = df.dropna(subset=["sentence", "acceptable"]).copy()

df["label"] = df["acceptable"].astype(int)   

In [69]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "val":  Dataset.from_pandas(val_df.reset_index(drop=True)),
})

train_ds, val_ds = ds["train"], ds["val"]

In [8]:
df["label"].value_counts(normalize=True)

label
1    0.745203
0    0.254797
Name: proportion, dtype: float64

In [9]:
model_name = "ai-forever/ruBert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tok(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = train_ds.map(tok, batched=True)
val_ds   = val_ds.map(tok, batched=True)

train_ds = train_ds.remove_columns(["sentence"])
val_ds   = val_ds.remove_columns(["sentence"])

train_ds.set_format("torch")
val_ds.set_format("torch")

config.json:   0%|          | 0.00/590 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/6295 [00:00<?, ? examples/s]

Map:   0%|          | 0/1574 [00:00<?, ? examples/s]

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

### добавим бОльший вес для наименее представленного класса (предложения, содержащие ошибки)

In [12]:
y = np.array(train_df["label"])
counts = np.bincount(y) 
class_weights = counts.sum() / (2.0 * counts)   
class_weights = torch.tensor(class_weights, dtype=torch.float)

class_weights

tensor([1.9623, 0.6710])

In [13]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.config.id2label = {0: "unacceptable", 1: "acceptable"}
model.config.label2id = {"unacceptable": 0, "acceptable": 1}

args = TrainingArguments(
    output_dir="rucola_rubert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.05,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    fp16=True,  
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()
trainer.evaluate()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/716M [00:00<?, ?B/s]

C:\Users\Lipidomics\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lipidomics\.cache\huggingface\hub\models--ai-forever--ruBert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of BertForSequenceClassification were not initialized from the model

model.safetensors:   0%|          | 0.00/716M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.650400,0.634407,0.734435,0.799841,0.858483,0.828125
2,0.517400,0.611972,0.762389,0.840580,0.840580,0.840580
3,0.332600,0.750939,0.774460,0.841402,0.859335,0.850274
4,0.245900,1.038396,0.796061,0.826687,0.919011,0.870408
5,0.148100,1.235211,0.792884,0.815809,0.932651,0.870326


C:\Users\Lipidomics\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\utils\generic.py:255: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  arr = np.array(obj)
C:\Users\Lipidomics\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\utils\generic.py:255: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.array(obj)
C:\Users\L

{'eval_loss': 1.038395643234253,
 'eval_accuracy': 0.7960609911054638,
 'eval_precision': 0.8266871165644172,
 'eval_recall': 0.9190110826939472,
 'eval_f1': 0.870407751312071,
 'eval_runtime': 77.7777,
 'eval_samples_per_second': 20.237,
 'eval_steps_per_second': 0.643,
 'epoch': 5.0}

### С увеличением числа эпох фнкция потерь для тренировочного датасета падает, а для валидационного сначала тоже падает, а затем увеличивается (с 3-ей эпохи), при этом метрики улучшаются. Это говорит о вероятном переобучении модели. В качестве компромисса между низкой функцией потерь и хорошими метриками выберем число эпох = 3.

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.config.id2label = {0: "unacceptable", 1: "acceptable"}
model.config.label2id = {"unacceptable": 0, "acceptable": 1}

args = TrainingArguments(
    output_dir="rucola_rubert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.05,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    fp16=True,  
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()
trainer.evaluate()

pytorch_model.bin:   0%|          | 0.00/716M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/716M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruBert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_55/1722524544.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.676200,0.650329,0.721728,0.796610,0.841432,0.818408
2,0.580800,0.593957,0.722999,0.839006,0.777494,0.807080


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.6503288149833679,
 'eval_accuracy': 0.7217280813214739,
 'eval_precision': 0.7966101694915254,
 'eval_recall': 0.8414322250639387,
 'eval_f1': 0.818407960199005,
 'eval_runtime': 7.7644,
 'eval_samples_per_second': 202.721,
 'eval_steps_per_second': 3.22,
 'epoch': 2.0}

In [17]:
test = pd.read_csv("/kaggle/input/rucola/test_rucola.csv")  
test = test.dropna(subset=["sentence", "acceptable"]).copy()

test["label"] = test["acceptable"].astype(int) 

In [26]:
test_ds = Dataset.from_pandas(test.reset_index(drop=True))
test_ds = test_ds.map(tok, batched=True)

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

In [27]:
test_metrics = trainer.evaluate(eval_dataset=test_ds)
print(test_metrics)

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.6369825005531311, 'eval_accuracy': 0.7375381485249237, 'eval_precision': 0.8088426527958388, 'eval_recall': 0.8485675306957708, 'eval_f1': 0.8282290279627164, 'eval_runtime': 5.1781, 'eval_samples_per_second': 189.837, 'eval_steps_per_second': 3.09, 'epoch': 2.0}


### RuGPT3

In [30]:
from transformers import AutoModelForCausalLM

gpt_name = "ai-forever/rugpt3small_based_on_gpt2"  
tokenizer_gpt = AutoTokenizer.from_pretrained(gpt_name)
model_gpt = AutoModelForCausalLM.from_pretrained(gpt_name)
model_gpt.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gpt.to(device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/551M [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50264, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50264, bias=False)
)

model.safetensors:   0%|          | 0.00/551M [00:00<?, ?B/s]

In [31]:
@torch.no_grad()
def score_completion(prompt: str, completion: str) -> float:

    prompt_ids = tokenizer_gpt(prompt, return_tensors="pt").input_ids.to(device)
    full_ids = tokenizer_gpt(prompt + completion, return_tensors="pt").input_ids.to(device)

    labels = full_ids.clone()
    labels[:, :prompt_ids.shape[1]] = -100

    out = model_gpt(full_ids, labels=labels)
    loss = out.loss.item()
    n_tokens = (labels != -100).sum().item()

    return -loss * n_tokens

### Zero shot

In [37]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

In [32]:
def make_zero_shot_prompt(sentence: str) -> str:
    return (
        "Задача: определить, грамматически приемлемо ли предложение на русском.\n"
        "Ответь строго одним символом: 1 (приемлемо) или 0 (есть ошибка).\n\n"
        f"Предложение: {sentence}\n"
        "Ответ:"
    )

In [41]:
def gpt_predict_label(prompt: str) -> int:
    s1 = score_completion(prompt, " 1")
    s0 = score_completion(prompt, " 0")
    return 1 if s1 > s0 else 0

def eval_zero_shot(df_eval: pd.DataFrame, limit=None):
    y_true = df_eval["label"].astype(int).to_numpy()
    if limit:
        df_eval = df_eval.iloc[:limit].copy()
        y_true = y_true[:limit]

    preds = []
    for sent in df_eval["sentence"].tolist():
        prompt = make_zero_shot_prompt(sent)
        preds.append(gpt_predict_label(prompt))

    preds = np.array(preds, dtype=int)
    return compute_metrics(y_true, preds)

In [42]:
### сделаем каждое предложение в тестовом датасете единичным промптом и посчитаем метрики
zero_shot = eval_zero_shot(test)

In [43]:
zero_shot

{'accuracy': 0.745676500508647,
 'precision': 0.745676500508647,
 'recall': 1.0,
 'f1': 0.8543123543123543}

### Few shots

In [52]:
def make_few_shot_prompt(sentence: str, exemples: list[tuple[str,int]]) -> str:
   
    head = (
        "Задача: определить, грамматически приемлемо ли предложение на русском.\n"
        "Ответь строго одним символом: 1 (приемлемо) или 0 (есть ошибка).\n\n"
        "Примеры:\n"
    )
    ex = "\n".join([f"Предложение: {t}\nОтвет: {y}" for t, y in exemples])
    tail = f"\n\nПредложение: {sentence}\nОтвет:"
    return head + ex + tail

In [56]:
import random

def eval_few_shot(train_df: pd.DataFrame, df_eval: pd.DataFrame, k=4, seed=42, limit=None):
    rng = random.Random(seed)

    y_true = df_eval["label"].astype(int).to_numpy()
    if limit:
        df_eval = df_eval.iloc[:limit].copy()
        y_true = y_true[:limit]
        
    pool = list(zip(train_df["sentence"].tolist(), train_df["label"].astype(int).tolist()))

    preds = []
    for sent in df_eval["sentence"].tolist():
        exemplars = rng.sample(pool, k)  
        prompt = make_few_shot_prompt(sent, exemplars)
        preds.append(gpt_predict_label(prompt))

    preds = np.array(preds, dtype=int)
    return compute_metrics(y_true, preds)

In [57]:
few_shot = eval_few_shot(train_df,test,k=1)

In [58]:
few_shot

{'accuracy': 0.6307222787385555,
 'precision': 0.7466666666666667,
 'recall': 0.7639836289222374,
 'f1': 0.7552258934592043}

In [59]:
few_shot_2 = eval_few_shot(train_df,test,k=2)

In [60]:
few_shot_2

{'accuracy': 0.7039674465920651,
 'precision': 0.7488738738738738,
 'recall': 0.9072305593451568,
 'f1': 0.8204811844540407}

In [61]:
few_shot_3 = eval_few_shot(train_df,test,k=3)

In [62]:
few_shot_3

{'accuracy': 0.6937945066124109,
 'precision': 0.7416107382550335,
 'recall': 0.9045020463847203,
 'f1': 0.8149969268592502}

In [64]:
few_shot_4 = eval_few_shot(train_df,test,k=4)

In [65]:
few_shot_4

{'accuracy': 0.6927772126144456,
 'precision': 0.7418630751964085,
 'recall': 0.9017735334242838,
 'f1': 0.8140394088669951}

### Метрики для генеративной модели сопоставимы с результатами файн-тьюнинга РуБЕРТ

### RuT5

In [67]:
from transformers import T5Tokenizer

POS_LABEL = "yes"
NEG_LABEL = "no"

model_name = "sberbank-ai/ruT5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.00M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [68]:
def preprocess_examples(examples):
    inputs = tokenizer(examples["sentence"], truncation=True)

    targets = [POS_LABEL if y == 1 else NEG_LABEL for y in examples["acceptable"]]
    labels = tokenizer(targets, truncation=True)["input_ids"]

    inputs["labels"] = labels
    return inputs

In [70]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "val":  Dataset.from_pandas(val_df.reset_index(drop=True)),
})

train_ds, val_ds = ds["train"], ds["val"]
test_ds = Dataset.from_pandas(test.reset_index(drop=True))

In [71]:
tokenized_train = train_ds.map(
    preprocess_examples,
    batched=True,
    remove_columns=train_ds.column_names
)

tokenized_val = val_ds.map(
    preprocess_examples,
    batched=True,
    remove_columns=val_ds.column_names
)

tokenized_test = test_ds.map(
    preprocess_examples,
    batched=True,
    remove_columns=test_ds.column_names
)

Map:   0%|          | 0/6295 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/1574 [00:00<?, ? examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

In [72]:
ACCURACY = evaluate.load("accuracy")
MCC = evaluate.load("matthews_correlation")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    pred_text = tokenizer.batch_decode(preds, skip_special_tokens=True)

    pred_int = np.array([1 if t.strip() == POS_LABEL else 0 for t in pred_text])

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    label_text = tokenizer.batch_decode(labels, skip_special_tokens=True)
    label_int = np.array([1 if t.strip() == POS_LABEL else 0 for t in label_text])

    return {
        "accuracy": ACCURACY.compute(predictions=pred_int, references=label_int)["accuracy"],
        "mcc": MCC.compute(predictions=pred_int, references=label_int)["matthews_correlation"],
    }

In [75]:
from transformers import (
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

model = T5ForConditionalGeneration.from_pretrained(model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)

args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/rucola",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=1e-4,           
    weight_decay=0.0,
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=3,        
    load_best_model_at_end=True,
    metric_for_best_model="eval_mcc",
    greater_is_better=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipykernel_55/2685389205.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Mcc
1,No log,0.205431,0.745235,0.000000
2,No log,0.195079,0.755400,0.165002
3,0.307300,0.198293,0.752859,0.255397
4,0.307300,0.206530,0.767471,0.262727
5,0.307300,0.214814,0.767471,0.258200


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.emb

TrainOutput(global_step=985, training_loss=0.24212242939750556, metrics={'train_runtime': 474.7341, 'train_samples_per_second': 66.3, 'train_steps_per_second': 2.075, 'total_flos': 1293106618782720.0, 'train_loss': 0.24212242939750556, 'epoch': 5.0})

In [80]:
test_metrics_ru5 = trainer.evaluate(eval_dataset=tokenized_test)

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [81]:
test_metrics_ru5

{'eval_loss': 0.2412276566028595,
 'eval_accuracy': 0.7934893184130214,
 'eval_mcc': 0.3703167325074006,
 'eval_runtime': 8.9686,
 'eval_samples_per_second': 109.604,
 'eval_steps_per_second': 1.784,
 'epoch': 5.0}